In [ ]:
import pandas as pd

In [12]:
#Premissas da análise
customer_id_cost = 10.00
gain = 0.30

In [5]:
#Número de usuários definidos no grupo target
#Lendo a tabela consumer_merge
consumer_merge = pd.read_parquet("../data/consumer_merge.parquet")
display(consumer_merge.head())
display(consumer_merge.info())
customer_id_target = consumer_merge[consumer_merge['is_target'] == 'target']['customer_id'].nunique()
display(customer_id_target)

,customer_id,is_target,active
0,755e1fa18f25caec5edffb188b13fd844b2af8cf5adedc...,target,True
1,b821aa8372b8e5b82cdc283742757df8c45eecdd72adf4...,control,True
2,d425d6ee4c9d4e211b71da8fc60bf6c5336b2ea9af9cc0...,control,True
3,6a7089eea0a5dc294fbccd4fa24d0d84a90c1cc12e829c...,target,True
4,dad6b7e222bab31c0332b0ccd9fa5dbd147008facd268f...,control,True


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 806466 entries, 0 to 806465
Data columns (total 3 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   customer_id  806466 non-null  object
 1   is_target    806466 non-null  object
 2   active       806156 non-null  object
dtypes: object(3)
memory usage: 18.5+ MB


None

445924

In [6]:
#Ticket médio do grupo target em dez/18 e jan/19
#Lendo a tabela ticket_medio_df
ticket_medio_df = pd.read_parquet("../data/ticket_medio_df.parquet")
display(ticket_medio_df.head())
display(ticket_medio_df.info())
ticket_medio_dez = ticket_medio_df.loc[0, 'target']
ticket_medio_jan = ticket_medio_df.loc[1, 'target']

,control,target
0,102.008186,109.495904
1,134.331655,151.886838


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   control  2 non-null      float64
 1   target   2 non-null      float64
dtypes: float64(2)
memory usage: 164.0 bytes


None

In [7]:
#Receita incremental por usuário
delta_ticket = ticket_medio_jan - ticket_medio_dez
receita_incremental_total = delta_ticket * customer_id_target

In [8]:
#Lucro incremental com margem de contribuição
lucro_incremental = receita_incremental_total * gain

In [9]:
#Custo da campanha
custo_total = customer_id_cost * customer_id_target

In [10]:
#ROI
roi = (lucro_incremental - custo_total) / custo_total

In [ ]:
import plotly.graph_objects as go

#Análise dos resultados
print(f"Número de usuários target: {customer_id_target}")
print(f"Ticket médio Dez/18: R$ {ticket_medio_dez:.2f}")
print(f"Ticket médio Jan/19: R$ {ticket_medio_jan:.2f}")
print(f"Receita incremental total: R$ {receita_incremental_total:,.2f}")
print(f"Lucro incremental (30% margem): R$ {lucro_incremental:,.2f}")
print(f"Custo total da campanha: R$ {custo_total:,.2f}")
print(f"ROI: {roi:.2%}")

#Resultado por gráfico
labels = ['Custo da Campanha', 'Receita Incremental', 'Lucro Incremental', 'ROI']
values = [-custo_total, receita_incremental_total, lucro_incremental, roi * 100]  # negativo para custo e positivo para receita/lucro

fig = go.Figure(go.Waterfall(
    x=labels,
    y=values,
    textposition="outside",
    increasing=dict(marker=dict(color="green")),
    decreasing=dict(marker=dict(color="red")),
    totals=dict(marker=dict(color="blue"))
))

fig.update_layout(
    title="Análise de Viabilidade Financeira - Gráfico de Caixinhas Sequenciais",
    xaxis_title="Métricas",
    yaxis_title="Valor (R$) / ROI (%)",
    template="plotly_white",
    showlegend=False
)

fig.show()

Número de usuários target: 445924
Ticket médio Dez/18: R$ 109.50
Ticket médio Jan/19: R$ 151.89
Receita incremental total: R$ 18,903,134.76
Lucro incremental (30% margem): R$ 5,670,940.43
Custo total da campanha: R$ 4,459,240.00
ROI: 27.17%
